# Agent Backtesting & Evaluation Framework

**AI Trader Pro — Quantitative Performance Analysis**

---

This notebook provides a rigorous backtesting and evaluation pipeline for the AI trading agents deployed on the *AI Trader Pro* platform. We simulate one calendar year of daily trading across five distinct algorithmic strategies, compute industry-standard risk-adjusted performance metrics, and apply statistical tests to characterise return distributions.

The analysis follows the methodology outlined in *Advances in Financial Machine Learning* (de Prado, 2018) and standard quantitative portfolio analytics conventions.

**Agents under evaluation:**

| Agent | Strategy | Expected Profile |
|---|---|---|
| MomentumBot | Trend-following | Moderate vol, positive drift |
| MeanReversionAgent | Statistical arbitrage | Low vol, mean-reverting |
| SentimentTrader | NLP / sentiment signals | High vol, episodic alpha |
| RiskParityAI | Risk-budgeted allocation | Very low vol, steady |
| AggressiveAlpha | Concentrated factor bets | High vol, fat tails |

## 1. Introduction & Setup

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({
    "figure.figsize": (14, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "font.family": "serif",
})

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TRADING_DAYS = 252
PALETTE = sns.color_palette("Set2", 5)

print(f"NumPy {np.__version__}  |  pandas {pd.__version__}")
print("Setup complete.")

## 2. Simulated Agent Portfolio Data

We generate synthetic daily return series for five agents, each calibrated to reflect realistic distributional characteristics of its stated strategy. All series span 252 trading days (one calendar year) and share a common random seed for reproducibility.

In [ ]:
rng = np.random.default_rng(seed=42)

agents = {
    "MomentumBot": {
        "strategy": "Trend-following",
        "returns": rng.normal(loc=0.0004, scale=0.012, size=TRADING_DAYS),
    },
    "MeanReversionAgent": {
        "strategy": "Statistical arbitrage",
        "returns": rng.normal(loc=0.0002, scale=0.007, size=TRADING_DAYS),
    },
    "SentimentTrader": {
        "strategy": "NLP / sentiment signals",
        "returns": rng.normal(loc=0.0003, scale=0.018, size=TRADING_DAYS)
                   + rng.choice([0, 0, 0, 0, 0.02, -0.015], size=TRADING_DAYS),
    },
    "RiskParityAI": {
        "strategy": "Risk-budgeted allocation",
        "returns": rng.normal(loc=0.00015, scale=0.004, size=TRADING_DAYS),
    },
    "AggressiveAlpha": {
        "strategy": "Concentrated factor bets",
        "returns": rng.standard_t(df=4, size=TRADING_DAYS) * 0.015 + 0.0003,
    },
}

returns_df = pd.DataFrame({name: a["returns"] for name, a in agents.items()})
returns_df.index.name = "trading_day"

print(f"Shape: {returns_df.shape}")
returns_df.describe().round(6)

## 3. Performance Metrics Implementation

Each metric is implemented as an independent, documented function following standard quantitative finance definitions. Annualisation assumes 252 trading days per year.

In [ ]:
def sharpe_ratio(returns: np.ndarray, risk_free_rate: float = 0.02) -> float:
    """Annualized Sharpe Ratio."""
    excess = returns - risk_free_rate / 252
    return np.sqrt(252) * excess.mean() / excess.std()


def sortino_ratio(returns: np.ndarray, risk_free_rate: float = 0.02) -> float:
    """Annualized Sortino Ratio (downside deviation only)."""
    excess = returns - risk_free_rate / 252
    downside = np.minimum(excess, 0)
    downside_std = np.sqrt(np.mean(downside**2))
    return np.sqrt(252) * excess.mean() / downside_std if downside_std > 0 else np.inf


def max_drawdown(returns: np.ndarray) -> float:
    """Maximum peak-to-trough drawdown."""
    cumulative = (1 + returns).cumprod()
    running_max = np.maximum.accumulate(cumulative)
    drawdowns = (cumulative - running_max) / running_max
    return drawdowns.min()


def win_rate(returns: np.ndarray) -> float:
    """Fraction of positive-return days."""
    return (returns > 0).sum() / len(returns)


def profit_factor(returns: np.ndarray) -> float:
    """Ratio of gross profits to gross losses."""
    gains = returns[returns > 0].sum()
    losses = abs(returns[returns < 0].sum())
    return gains / losses if losses > 0 else np.inf


def calmar_ratio(returns: np.ndarray) -> float:
    """Annualized return / |max drawdown|."""
    annual_return = (1 + returns).prod() ** (252 / len(returns)) - 1
    mdd = abs(max_drawdown(returns))
    return annual_return / mdd if mdd > 0 else np.inf


def value_at_risk(returns: np.ndarray, confidence: float = 0.95) -> float:
    """Historical VaR at given confidence level."""
    return np.percentile(returns, (1 - confidence) * 100)


def conditional_var(returns: np.ndarray, confidence: float = 0.95) -> float:
    """Expected Shortfall (CVaR) beyond the VaR threshold."""
    var = value_at_risk(returns, confidence)
    return returns[returns <= var].mean()


print("All metric functions defined.")

## 4. Comprehensive Metrics Table

We evaluate every agent across the full metric suite and present the results as a styled DataFrame with conditional formatting.

In [ ]:
records = []
for name, data in agents.items():
    r = data["returns"]
    annual_ret = (1 + r).prod() ** (252 / len(r)) - 1
    annual_vol = r.std() * np.sqrt(252)
    records.append({
        "Agent": name,
        "Strategy": data["strategy"],
        "Annual Return %": round(annual_ret * 100, 2),
        "Annual Volatility %": round(annual_vol * 100, 2),
        "Sharpe": round(sharpe_ratio(r), 3),
        "Sortino": round(sortino_ratio(r), 3),
        "Max Drawdown %": round(max_drawdown(r) * 100, 2),
        "Win Rate %": round(win_rate(r) * 100, 1),
        "Profit Factor": round(profit_factor(r), 3),
        "Calmar": round(calmar_ratio(r), 3),
        "VaR 95%": round(value_at_risk(r) * 100, 4),
        "CVaR 95%": round(conditional_var(r) * 100, 4),
    })

metrics_df = pd.DataFrame(records).set_index("Agent")

numeric_cols = metrics_df.select_dtypes(include="number").columns
styled = (
    metrics_df.style
    .background_gradient(cmap="RdYlGn", subset=numeric_cols)
    .format(precision=3, subset=numeric_cols)
    .set_caption("Agent Performance Dashboard")
)
styled

## 5. Visualizations

### 5a. Cumulative Returns

In [ ]:
cumulative = (1 + returns_df).cumprod()

fig, ax = plt.subplots(figsize=(14, 6))
for i, col in enumerate(cumulative.columns):
    ax.plot(cumulative[col], label=col, color=PALETTE[i], linewidth=1.4)
ax.axhline(1.0, color="grey", linestyle="--", linewidth=0.8)
ax.set_title("Cumulative Returns by Agent")
ax.set_xlabel("Trading Day")
ax.set_ylabel("Growth of $1")
ax.legend(loc="upper left", frameon=True)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "cumulative_returns.png", dpi=150, bbox_inches="tight")
plt.show()

### 5b. Drawdown Analysis

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=True)

for i, col in enumerate(returns_df.columns):
    cum = (1 + returns_df[col]).cumprod()
    running_max = cum.cummax()
    dd = (cum - running_max) / running_max

    axes[i].fill_between(dd.index, dd.values, 0, color="crimson", alpha=0.35)
    axes[i].plot(dd, color="darkred", linewidth=0.8)
    axes[i].set_ylabel("Drawdown")
    axes[i].set_title(f"{col}  (Max DD: {dd.min():.2%})")
    axes[i].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

axes[-1].set_xlabel("Trading Day")
fig.suptitle("Drawdown Curves", fontsize=16, y=1.01)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "drawdown_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

### 5c. Risk-Return Scatter

In [ ]:
ann_rets = [(1 + returns_df[c]).prod() ** (252 / TRADING_DAYS) - 1 for c in returns_df]
ann_vols = [returns_df[c].std() * np.sqrt(252) for c in returns_df]
sharpes  = [sharpe_ratio(returns_df[c].values) for c in returns_df]

fig, ax = plt.subplots(figsize=(10, 7))

vol_range = np.linspace(0.01, max(ann_vols) * 1.3, 200)
ef_ret = 0.02 + 0.6 * vol_range
ax.plot(vol_range * 100, ef_ret * 100, "--", color="grey", alpha=0.5, label="CML reference")

for i, col in enumerate(returns_df.columns):
    size = max(abs(sharpes[i]) * 200, 60)
    ax.scatter(ann_vols[i] * 100, ann_rets[i] * 100, s=size,
               color=PALETTE[i], edgecolors="black", linewidth=0.6, zorder=5)
    ax.annotate(col, (ann_vols[i] * 100, ann_rets[i] * 100),
                textcoords="offset points", xytext=(8, 6), fontsize=9)

ax.set_xlabel("Annualized Volatility (%)")
ax.set_ylabel("Annualized Return (%)")
ax.set_title("Risk-Return Profile (bubble size proportional to |Sharpe|)")
ax.legend(loc="upper left")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "risk_return_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

### 5d. Rolling Sharpe Ratio (60-day)

In [ ]:
WINDOW = 60

fig, ax = plt.subplots(figsize=(14, 6))
for i, col in enumerate(returns_df.columns):
    excess = returns_df[col] - 0.02 / 252
    rolling_sharpe = (
        excess.rolling(WINDOW).mean() / excess.rolling(WINDOW).std()
    ) * np.sqrt(252)
    ax.plot(rolling_sharpe, label=col, color=PALETTE[i], linewidth=1.2)

ax.axhline(0, color="black", linewidth=0.6)
ax.set_title(f"Rolling Sharpe Ratio ({WINDOW}-day window)")
ax.set_xlabel("Trading Day")
ax.set_ylabel("Sharpe Ratio (annualized)")
ax.legend(loc="lower right", frameon=True)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "rolling_sharpe.png", dpi=150, bbox_inches="tight")
plt.show()

### 5e. Return Distributions

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 5), sharey=True)

for i, col in enumerate(returns_df.columns):
    data = returns_df[col]
    axes[i].hist(data, bins=40, density=True, alpha=0.5, color=PALETTE[i], edgecolor="white")
    sns.kdeplot(data, ax=axes[i], color=PALETTE[i], linewidth=1.5, label="KDE")

    x = np.linspace(data.min(), data.max(), 200)
    axes[i].plot(x, stats.norm.pdf(x, data.mean(), data.std()),
                 "k--", linewidth=1, label="Normal")

    sk = stats.skew(data)
    ku = stats.kurtosis(data)
    axes[i].annotate(f"skew={sk:.2f}\nkurt={ku:.2f}",
                     xy=(0.05, 0.90), xycoords="axes fraction",
                     fontsize=8, verticalalignment="top",
                     bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("Daily Return")
    if i == 0:
        axes[i].set_ylabel("Density")
    axes[i].legend(fontsize=7)

fig.suptitle("Return Distributions vs. Normal", fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "return_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

### 5f. Correlation Heatmap

In [ ]:
corr = returns_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            mask=mask, square=True, linewidths=0.5, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Agent Return Correlations")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Statistical Tests

We apply the **Jarque-Bera test** ($H_0$: returns are normally distributed) to each agent's daily return series. Rejection at $\alpha = 0.05$ indicates significant non-normality — a common finding in financial returns and especially relevant for tail-risk management.

In [ ]:
jb_results = []
for col in returns_df.columns:
    jb_stat, p_val = stats.jarque_bera(returns_df[col])
    jb_results.append({
        "Agent": col,
        "JB Statistic": round(jb_stat, 3),
        "p-value": f"{p_val:.4e}",
        "Normal at 5%?": "Yes" if p_val > 0.05 else "No",
        "Interpretation": (
            "Cannot reject normality" if p_val > 0.05
            else "Significant departure from normality"
        ),
    })

jb_df = pd.DataFrame(jb_results).set_index("Agent")
jb_df

**Observation:** Agents whose return-generating processes include fat tails (e.g., Student-$t$ innovations in *AggressiveAlpha*) or mixture components (e.g., sentiment-driven spikes in *SentimentTrader*) tend to strongly reject the normality hypothesis. Standard Gaussian VaR therefore *underestimates* tail risk for these strategies.

## 7. Ranking & Conclusions

We construct a **composite score** from five normalised metrics, weighted to reflect a balanced preference for risk-adjusted return, downside protection, and consistency:

| Metric | Weight | Rationale |
|---|---|---|
| Sharpe Ratio | 30% | Primary risk-adjusted return measure |
| Sortino Ratio | 20% | Penalises downside volatility specifically |
| Max Drawdown | 20% | Capital preservation proxy |
| Win Rate | 15% | Strategy consistency |
| Profit Factor | 15% | Reward-to-pain ratio |

In [ ]:
ranking_data = []
for name, data in agents.items():
    r = data["returns"]
    ranking_data.append({
        "Agent": name,
        "Sharpe": sharpe_ratio(r),
        "Sortino": sortino_ratio(r),
        "MaxDD": -max_drawdown(r),
        "WinRate": win_rate(r),
        "ProfitFactor": profit_factor(r),
    })

rank_df = pd.DataFrame(ranking_data).set_index("Agent")

norm = (rank_df - rank_df.min()) / (rank_df.max() - rank_df.min() + 1e-9)

weights = {
    "Sharpe": 0.30,
    "Sortino": 0.20,
    "MaxDD": 0.20,
    "WinRate": 0.15,
    "ProfitFactor": 0.15,
}

norm["Composite Score"] = sum(norm[k] * w for k, w in weights.items())
norm = norm.sort_values("Composite Score", ascending=False)
norm["Rank"] = range(1, len(norm) + 1)

norm.style.background_gradient(
    cmap="YlGn", subset=["Composite Score"]
).format(precision=3).set_caption("Agent Ranking by Composite Score")

In [ ]:
print("=" * 60)
print("           FINAL AGENT RANKING")
print("=" * 60)
for _, row in norm.iterrows():
    print(f"  #{int(row['Rank'])}  {row.name:<25s}  Score: {row['Composite Score']:.3f}")
print("=" * 60)

### Qualitative Conclusions

1. **Risk-adjusted returns dominate raw returns.** Agents with lower volatility (e.g., *RiskParityAI*, *MeanReversionAgent*) rank highly despite modest absolute returns, because their Sharpe and Sortino ratios benefit from tighter dispersion.

2. **Tail risk is not captured by volatility alone.** *AggressiveAlpha* exhibits the widest return distribution and the heaviest tails (highest excess kurtosis), making standard deviation an insufficient risk measure. The Jarque-Bera test confirms significant non-normality for this agent, reinforcing the need for CVaR-based risk budgeting.

3. **Drawdown discipline separates survivors from casualties.** Even agents with high annual returns can score poorly if they experience deep or prolonged drawdowns, reflecting the real-world constraint that most allocators apply drawdown limits.

4. **Diversification potential exists.** The correlation heatmap reveals that several agent pairs have near-zero correlation, suggesting that an ensemble portfolio of these agents could achieve superior risk-adjusted returns through diversification — a natural extension of this analysis.

5. **Sentiment-driven strategies are inherently regime-dependent.** The episodic alpha generated by *SentimentTrader* manifests as high kurtosis and an unstable rolling Sharpe, making it better suited as a tactical overlay rather than a core allocation.

---

*This evaluation framework is part of the AI Trader Pro platform. All data is synthetic and intended for demonstration purposes.*